# M01.01 — Foundations

Everything the rest of the day builds — memory, tools, agents — is **orchestration we write
around a stateless LLM call**. This notebook is a live demo, not a hands-on exercise: it exists
to make that one idea concrete before we start layering things on top of it in M02.

In [ ]:
from agentic_workshop.client import get_client
from agentic_workshop.config import CHAT_MODEL, REASONING_OFF

client = get_client()
MODEL = CHAT_MODEL

## The Model Is Stateless

`client.responses.create()` takes an `input` list of messages and returns one response.

There is no intrinsic connection between two calls that we make to the model.
The model doesn't know or care if the last message in a conversation is from 5 seconds ago or 5 hours.


In [ ]:
response = client.responses.create(
    model=MODEL,
    input=[{"role": "user", "content": "My name is Alex. Remember that."}],
    reasoning=REASONING_OFF,
)
print(response.output_text)


In [ ]:
response = client.responses.create(
    model=MODEL,
    input=[{"role": "user", "content": "What is my name?"}],
    reasoning=REASONING_OFF,
)
print(response.output_text)


Unlike general world knowledge, particular knowledge like this is only available to the LLM if we explicitly give it.

We need to provide the model with this information if we want it to use it to respond with.

## Memory Management is a Layer on Top

Let's build the minimal version of that ourselves: a growing `input` list that we pass in
full on every turn.


In [ ]:
# Initialize the conversation with our first message
conversation = [{"role": "user", "content": "My name is Alex. Remember that."}]

In [ ]:
# Get a response and add it to the conversation
response = client.responses.create(model=MODEL, input=conversation, reasoning=REASONING_OFF)
conversation.append({"role": "assistant", "content": response.output_text})
print(response.output_text)


You'll notice that we mark the LLM's response as an `assistant` message.

Distinguishing where each piece of information came from is useful when we build the context we give a model: most LLMs are trained on conversations between a `user` and an `assistant`.

A third role, `system` (OpenAI also calls it `developer`), holds the instructions on how the model should behave.

In [ ]:
# Our turn again
conversation.append({"role": "user", "content": "What is my name?"})

In [ ]:
# Back to the model
response = client.responses.create(model=MODEL, input=conversation, reasoning=REASONING_OFF)
conversation.append({"role": "assistant", "content": response.output_text})
print(response.output_text)


Same API, same stateless call — the only difference is *we* now carry the history. This is the
entire idea behind "memory" in LLM applications. Everything else we'll build today (retrieval,
tool calling, agent loops) is the same trick: decide what to put in `input` before calling
the model.

As a reminder, see below what we sent the model before it could answer with the correct name:

In [ ]:
for msg in conversation[:-1]:
    print(msg)

## Tokens and the context window

`input` isn't free or infinite. The model reads it as **tokens**, and every provider caps how
many tokens fit in a single call (the **context window**). A growing conversation costs more,
and eventually stops fitting at all.


In [ ]:
import tiktoken

# o200k_base is the tokenizer of recent OpenAI models. It is an approximation for models tiktoken doesn't know by name.
tokenizer = tiktoken.get_encoding("o200k_base")

def count_tokens(messages):
    return sum(len(tokenizer.encode(m["content"])) for m in messages)

print(f"Tokens in conversation so far: {count_tokens(conversation)}")
for m in conversation:
    print(f"  [{m['role']:>9}] {len(tokenizer.encode(m['content'])):>3} tokens  -  {m['content']!r}")

Pricing is per call. We called the model twice, and each time we sent everything from before plus the new message. Over many turns, this repeated input gets expensive.

In [ ]:
from agentic_workshop.costs import PRICES

steps = [(count_tokens(conversation[:i]), count_tokens([m])) for i, m in enumerate(conversation) if m["role"] == "assistant"]
for n, (i, o) in enumerate(steps, 1):
    print(f"Call {n}: input {i:>4}  output {o:>4}")
total_in, total_out = sum(i for i, _ in steps), sum(o for _, o in steps)
cost = (total_in * PRICES[MODEL]["input"] + total_out * PRICES[MODEL]["output"]) / 1e6
print(f"Total:  input {total_in:>4}  output {total_out:>4}  (~${cost:.6f}, message text only)")

This conversation is tiny, but imagine pasting an entire PDF into `input` on every turn, or a very long conversation. We come back to this in M03 (RAG).

## Latency

Making a request from the LLM takes significant time. Every call costs us a network round trip plus generation time. This becomes especially relevant once you need multiple LLM responses chained together to answer a user's query. Additionally, longer prompts and longer outputs both cost time as well as money.

In [ ]:
import time

start = time.perf_counter()
first_token_time = None
chunk_times = []
text = ""
usage = None

with client.responses.create(
    model=MODEL,
    input=[{"role": "user", "content": "In one sentence, what is a context window?"}],
    reasoning=REASONING_OFF,
    stream=True,
) as stream:
    for event in stream:
        if event.type == "response.output_text.delta":
            now = time.perf_counter()
            if first_token_time is None:
                first_token_time = now - start
            chunk_times.append(now)
            text += event.delta
            print(event.delta, end="", flush=True)
        elif event.type == "response.completed":
            usage = event.response.usage

total_time = time.perf_counter() - start
generation_time = chunk_times[-1] - chunk_times[0]
visible_tokens = len(tokenizer.encode(text))

print("\n")
print(f"Time to first token: {first_token_time:.2f} s")
print(f"Total time:          {total_time:.2f} s")
print(f"Visible output tokens: {visible_tokens}")
if usage is not None:
    print(f"Billed output tokens:  {usage.output_tokens} (of which reasoning: {usage.output_tokens_details.reasoning_tokens})")
if visible_tokens > 1 and generation_time > 0:
    print(f"Tokens per second:   {(visible_tokens - 1) / generation_time:.1f}")

Notice how much of the total time passes before the first token arrives. Part of it is network overhead. The rest is the model processing the input (and, for a reasoning model, thinking) before it writes anything. The total time is composed from network overhead, model processing, and the stream finishing.

The API also bills more output tokens than we see. Reasoning models generate **reasoning tokens** before the answer: they are not always returned to you, but you pay for them as output tokens.

In [ ]:
start = time.perf_counter()
response = client.responses.create(
    model=MODEL,
    input=[{"role": "user", "content": "In one sentence, what is a context window?"}],
)

total_time = time.perf_counter() - start
visible_tokens = len(tokenizer.encode(response.output_text))

print(response.output_text)
print("\n")
print(f"Total time:            {total_time:.2f} s")
print(f"Visible output tokens: {visible_tokens}")
print(f"Billed output tokens:  {response.usage.output_tokens} (of which reasoning: {response.usage.output_tokens_details.reasoning_tokens})")

You'll notice now that since we did not suppress the reasoning, we got much more invisible output!

Additionally if you look at the amounts now, rather than 4 extra tokens like before we're now at 10!

These extra tokens are likely control tokens for the model to switch between reasoning and visible token generation, since every action the model takes is also just generated tokens.

# Takeaways

- The model is stateless and doesn't track conversation history
- The management of history and the context that we provide the model is entirely up to us
- The context window is what provides the limit for how much we can show a langauge model at once
- Time to first token (prompt processing) takes longer with larger prompts and is never free or instant (can become cheaper with caching)
- Reasoning models output tokens that are not shown but we are still billed for them



## Where we're going next

Everything today is a variation on "decide what goes into `input` before calling the model":

- **M2 — Structured Output & Tool Calling**: instead of free text back, we constrain what comes
  *out* of the model so other code can consume it directly.
- **M3 — RAG**: instead of pasting a whole document in, we retrieve only the relevant pieces and
  put *those* into `input`.
- **M4 — Agentic Patterns**: instead of us deciding the sequence of calls, the model decides
  which tool to call next, in a loop.
- **M5 — Production Considerations**: how do we know any of this is actually working?
- **M6 — AI for Code Development**: the same stateless-call-plus-orchestration idea, applied to
  a coding agent editing a real codebase.
